# Team 2 - Feature Extraction
### Member: Aasim
### Project: Poaching Threat Detection | SDG Goal 15 — Life on Land

In [ ]:
import os
import numpy as np
import pandas as pd
import librosa
from tqdm import tqdm

SAMPLE_RATE = 44100
N_FFT       = 2048
HOP_LENGTH  = 512

CSV_OUTPUT = 'metadata.csv'
df = pd.read_csv(CSV_OUTPUT)
print(f'Loaded {len(df)} files')


In [ ]:
def extract_spectral_features(audio, sr=SAMPLE_RATE):
    """
    Extracts spectral features that specifically help distinguish
    mechanical threat sounds from natural wildlife sounds.

    Why these features matter for poaching detection:
      - Spectral Centroid  : Mechanical sounds (chainsaw, gunshot) push
                             energy to higher frequencies → high centroid
      - Spectral Bandwidth : Mechanical sounds have consistent narrow
                             bandwidth unlike varied natural sounds
      - Spectral Rolloff   : Illegal tools push energy to high rolloff point
      - Spectral Contrast  : Peaks and valleys differ for each sound type
      - ZCR                : Gunshots and chainsaws have very high ZCR
      - RMS Energy         : Threat sounds are typically louder

    Features extracted:
      - Spectral Centroid  (mean + std) →  2 features
      - Spectral Bandwidth (mean + std) →  2 features
      - Spectral Rolloff   (mean + std) →  2 features
      - Spectral Contrast  (mean + std, 7 bands) → 14 features
      - Zero Crossing Rate (mean + std) →  2 features
      - RMS Energy         (mean + std) →  2 features
    ─────────────────────────────────────────────────────────
    Subtotal (Aasim)                   → 24 features
    """
    features = {}

    # ── Spectral Centroid ──────────────────────────────────
    centroid = librosa.feature.spectral_centroid(y=audio, sr=sr,
                                                  n_fft=N_FFT, hop_length=HOP_LENGTH)
    features['spectral_centroid_mean'] = np.mean(centroid)
    features['spectral_centroid_std']  = np.std(centroid)

    # ── Spectral Bandwidth ─────────────────────────────────
    bandwidth = librosa.feature.spectral_bandwidth(y=audio, sr=sr,
                                                    n_fft=N_FFT, hop_length=HOP_LENGTH)
    features['spectral_bandwidth_mean'] = np.mean(bandwidth)
    features['spectral_bandwidth_std']  = np.std(bandwidth)

    # ── Spectral Rolloff ───────────────────────────────────
    rolloff = librosa.feature.spectral_rolloff(y=audio, sr=sr,
                                                n_fft=N_FFT, hop_length=HOP_LENGTH)
    features['spectral_rolloff_mean'] = np.mean(rolloff)
    features['spectral_rolloff_std']  = np.std(rolloff)

    # ── Spectral Contrast ──────────────────────────────────
    # 7 frequency bands — each with mean and std
    contrast = librosa.feature.spectral_contrast(y=audio, sr=sr,
                                                  n_fft=N_FFT, hop_length=HOP_LENGTH)
    for i in range(contrast.shape[0]):
        features[f'spectral_contrast_{i+1}_mean'] = np.mean(contrast[i])
        features[f'spectral_contrast_{i+1}_std']  = np.std(contrast[i])

    # ── Zero Crossing Rate ─────────────────────────────────
    zcr = librosa.feature.zero_crossing_rate(y=audio, hop_length=HOP_LENGTH)
    features['zcr_mean'] = np.mean(zcr)
    features['zcr_std']  = np.std(zcr)

    # ── RMS Energy ─────────────────────────────────────────
    rms = librosa.feature.rms(y=audio, hop_length=HOP_LENGTH)
    features['rms_mean'] = np.mean(rms)
    features['rms_std']  = np.std(rms)

    return features

print('extract_spectral_features() function defined!')
print('Spectral features covered by Aasim: Centroid + Bandwidth + Rolloff + Contrast + ZCR + RMS = 24 features')
